# 06 - Azure ML Studio Guide

**Goal:** Understand how Azure ML Studio works so you can set it up when ready. This notebook is a reference guide — code examples to try when you connect to your Azure account.

---

## Azure ML Studio vs DVC + MLflow

Azure ML Studio does the same things as DVC + MLflow, but as a managed service:

```
DVC + MLflow (open source)         Azure ML Studio (managed)
┌─────────────────────┐            ┌─────────────────────────┐
│ DVC: data versioning│            │ Data Assets: versioning │
│ MLflow: tracking    │            │ MLflow: tracking (same!)│
│ MLflow: registry    │            │ Model Registry: built-in│
│ DVC: pipelines      │            │ Pipelines: SDK v2       │
│ Your servers        │            │ Managed compute         │
│ Your UI (MLflow)    │            │ Azure Portal UI         │
└─────────────────────┘            └─────────────────────────┘
```

**Key insight:** Azure ML uses MLflow natively for experiment tracking. So your MLflow code from notebooks 02-03 works in Azure ML with zero changes — you just point `mlflow.set_tracking_uri()` to Azure.

## Architecture Overview

```
Azure ML Workspace
├── Datastores          ← connections to Blob Storage, ADLS, etc.
├── Data Assets          ← versioned references to data (like DVC pointers)
├── Experiments          ← groups of runs (MLflow experiments)
│   └── Runs             ← individual training runs (MLflow runs)
├── Models               ← model registry (like MLflow registry)
├── Pipelines            ← multi-step workflows (like DVC pipelines)
├── Compute              ← VMs, clusters, serverless
└── Endpoints            ← deployed models (inference APIs)
```

Everything lives under a **Workspace**, which is like a project container.

## Setup: What You Need

```bash
# Install the SDK v2 (not the old azureml-core v1)
pip install azure-ai-ml azure-identity mlflow azureml-mlflow
```

| Package | Purpose |
|---------|--------|
| `azure-ai-ml` | SDK v2 — data, models, pipelines, compute |
| `azure-identity` | Authentication (login) |
| `mlflow` | Experiment tracking (you already have this) |
| `azureml-mlflow` | Plugin that connects MLflow → Azure ML backend |

## 1. Connect to Your Workspace

The central object is `MLClient`. Everything goes through it.

In [ ]:
# === THIS CODE REQUIRES AZURE CREDENTIALS ===
# Run this when you're connected to your Azure account.
# For now, read through it as a reference.

# from azure.ai.ml import MLClient
# from azure.identity import DefaultAzureCredential

# ml_client = MLClient(
#     credential=DefaultAzureCredential(),
#     subscription_id="<YOUR_SUBSCRIPTION_ID>",
#     resource_group_name="<YOUR_RESOURCE_GROUP>",
#     workspace_name="<YOUR_WORKSPACE_NAME>",
# )

# # Or from a config.json (download from Azure Portal)
# ml_client = MLClient.from_config(credential=DefaultAzureCredential())

print("Azure ML code is commented out — uncomment when connected to Azure.")

### Authentication

`DefaultAzureCredential` tries multiple auth methods in order:

| Priority | Method | When to use |
|----------|--------|------------|
| 1 | Environment variables | CI/CD pipelines, automated jobs |
| 2 | Managed Identity | Running on Azure VMs or compute |
| 3 | Azure CLI (`az login`) | Local development |
| 4 | Interactive browser | Fallback |

For local development, just run `az login` first and it works.

## 2. Data Assets (Replaces DVC)

Data Assets are versioned references to your data. Like `.dvc` files, but managed by Azure.

Three types:

| Type | Use case | Example |
|------|----------|--------|
| `URI_FILE` | Single file | One CSV, one NIfTI file |
| `URI_FOLDER` | Folder of files | A dataset of CT scans |
| `MLTABLE` | Tabular with schema | Patient demographics CSV |

In [ ]:
# === AZURE ML: Register a dataset ===

# from azure.ai.ml.entities import Data
# from azure.ai.ml.constants import AssetTypes

# # Register spine CT scans (a folder of NIfTI files)
# spine_data = Data(
#     name="spine-ct-scans",
#     version="1",
#     description="200 spine CT scans, annotated for vertebra segmentation",
#     path="azureml://datastores/workspaceblobstore/paths/data/spine_scans/",
#     type=AssetTypes.URI_FOLDER,
#     tags={"modality": "CT", "body_part": "spine", "num_patients": "200"},
# )
# ml_client.data.create_or_update(spine_data)

# # Register version 2 (more patients)
# spine_data_v2 = Data(
#     name="spine-ct-scans",
#     version="2",
#     description="400 spine CT scans (added 200 new patients)",
#     path="azureml://datastores/workspaceblobstore/paths/data/spine_scans_v2/",
#     type=AssetTypes.URI_FOLDER,
#     tags={"modality": "CT", "body_part": "spine", "num_patients": "400"},
# )
# ml_client.data.create_or_update(spine_data_v2)

print("Data Assets = versioned, immutable references to your data.")
print("Once created, a version CANNOT be modified or deleted.")
print("This is critical for FDA traceability.")

In [ ]:
# === AZURE ML: Retrieve data ===

# # Get specific version
# data = ml_client.data.get(name="spine-ct-scans", version="2")
# print(f"Name: {data.name}")
# print(f"Version: {data.version}")
# print(f"Path: {data.path}")
# print(f"Tags: {data.tags}")

# # List all versions
# for d in ml_client.data.list(name="spine-ct-scans"):
#     print(f"  v{d.version}: {d.description}")

print("Data Assets vs DVC:")
print("  DVC:   .dvc file in Git  →  points to Azure Blob")
print("  Azure: Data Asset in UI  →  points to Azure Blob")
print("  Same storage, different catalog.")

### Data Assets vs DVC

| Feature | DVC | Azure Data Assets |
|---------|-----|------------------|
| Where catalog lives | Git repo (`.dvc` files) | Azure ML Workspace |
| Where data lives | Azure Blob (same!) | Azure Blob (same!) |
| Immutable versions | Via git commits | Built-in (can't edit/delete) |
| Browse versions | `git log` | Azure Portal UI |
| Tags / metadata | Limited | Rich (tags, description) |
| Works offline | Yes | No (needs Azure connection) |
| Cost | Free | Free (part of workspace) |

## 3. Experiment Tracking (Same MLflow!)

This is the best part: **Azure ML uses MLflow natively**. Your tracking code doesn't change — you just point MLflow to Azure.

```python
# LOCAL (what you learned in notebook 02):
mlflow.set_tracking_uri("file:///tmp/mlflow-demo")

# AZURE ML (just change the URI):
mlflow.set_tracking_uri(ml_client.workspaces.get(ws_name).mlflow_tracking_uri)

# Everything else is IDENTICAL:
mlflow.set_experiment("spine-segmentation")
with mlflow.start_run():
    mlflow.log_param("lr", 0.001)
    mlflow.log_metric("dice", 0.89)
    mlflow.pytorch.log_model(model, "model")
```

Zero code changes. That's why we learned MLflow first.

In [ ]:
# === AZURE ML: Connect MLflow to workspace ===

# import mlflow

# # Get the tracking URI from the workspace
# tracking_uri = ml_client.workspaces.get(
#     ml_client.workspace_name
# ).mlflow_tracking_uri
# 
# mlflow.set_tracking_uri(tracking_uri)
#
# # Now ALL your MLflow code from notebooks 02 and 03 works against Azure.
# # No changes needed:
# mlflow.set_experiment("spine-segmentation")
# with mlflow.start_run(run_name="swinunetr-v9"):
#     mlflow.log_param("lr", 0.001)
#     mlflow.log_param("features", 24)
#     mlflow.log_metric("val_dice", 0.891)
#     mlflow.pytorch.log_model(model, "model")

print("Azure ML tracking = MLflow tracking.")
print("Your code from notebook 02 works with zero changes.")
print("Just change the tracking URI.")

### On Azure ML Compute: Even Simpler

When your training script runs on Azure ML compute (a VM in the cloud), MLflow is **auto-configured**. No `set_tracking_uri()` needed:

```python
# training_semantic.py running on Azure ML compute
import mlflow

# Just log — Azure ML handles the rest
mlflow.log_param("lr", 0.001)
mlflow.log_metric("dice", 0.89)
```

## 4. Model Registry

Azure ML has its own model registry (similar to MLflow's, but integrated with deployment).

In [ ]:
# === AZURE ML: Register a model ===

# from azure.ai.ml.entities import Model
# from azure.ai.ml.constants import AssetTypes

# # Option A: From a local file
# model = Model(
#     path="./weights/best_metric_model.pth",
#     type=AssetTypes.CUSTOM_MODEL,
#     name="spine-semantic-swinunetr",
#     version="1",
#     description="SwinUNETR region model. Dice=0.891. Data v2 (400 patients).",
#     tags={"architecture": "SwinUNETR", "dice": "0.891", "data_version": "v2"},
# )
# ml_client.models.create_or_update(model)

# # Option B: From an MLflow run (best practice)
# model = Model(
#     path=f"runs:/{run_id}/model",
#     type=AssetTypes.MLFLOW_MODEL,
#     name="spine-semantic-swinunetr",
#     version="2",
#     description="Improved model from experiment run.",
# )
# ml_client.models.create_or_update(model)

print("Model Registry:")
print("  - Versioned (v1, v2, v3...)")
print("  - Tagged (architecture, dice, data_version)")
print("  - Can be deployed to endpoints directly")
print("  - Archived when retired")

In [ ]:
# === AZURE ML: Model lifecycle management ===

# # List all versions
# for m in ml_client.models.list(name="spine-semantic-swinunetr"):
#     print(f"  v{m.version}: {m.description} | tags={m.tags}")

# # Tag as production (Azure ML uses tags, not stages like MLflow)
# model = ml_client.models.get(name="spine-semantic-swinunetr", version="2")
# model.tags["stage"] = "Production"
# ml_client.models.create_or_update(model)

# # Archive old version
# ml_client.models.archive(name="spine-semantic-swinunetr", version="1")

print("Azure ML Model Registry vs MLflow Model Registry:")
print("  MLflow:    stages (None/Staging/Production/Archived)")
print("  Azure ML:  tags + archive (more flexible)")
print("  Both:      versioned, traceable back to training run")

## 5. Pipelines (Replaces DVC Pipelines)

Azure ML Pipelines use Python SDK v2 with the `@dsl.pipeline` decorator.

```
DVC Pipeline (dvc.yaml):              Azure ML Pipeline (@dsl.pipeline):
┌──────────────────────┐              ┌──────────────────────────┐
│ stages:              │              │ @dsl.pipeline            │
│   prepare_data:      │              │ def training_pipeline(): │
│     cmd: python ...  │              │   step1 = prepare(...)   │
│   train:             │              │   step2 = train(...)     │
│     cmd: python ...  │              │   step3 = evaluate(...)  │
│   evaluate:          │              │                          │
│     cmd: python ...  │              │                          │
└──────────────────────┘              └──────────────────────────┘
Runs locally or on any machine        Runs on Azure compute (cloud)
```

In [ ]:
# === AZURE ML: Define a training pipeline ===

# from azure.ai.ml import dsl, Input, command

# # Define pipeline steps as components
# prepare_step = command(
#     name="prepare_data",
#     display_name="Prepare Spine Data",
#     inputs={"raw_data": Input(type="uri_folder")},
#     outputs={"processed_data": Output(type="uri_folder")},
#     code="./src",
#     command="python prepare_data.py --input ${{inputs.raw_data}} --output ${{outputs.processed_data}}",
#     environment="AzureML-pytorch-2.0@latest",
# )

# train_step = command(
#     name="train_model",
#     display_name="Train SwinUNETR",
#     inputs={
#         "training_data": Input(type="uri_folder"),
#         "config": Input(type="uri_file"),
#     },
#     outputs={"model": Output(type="uri_folder")},
#     code="./src",
#     command="python training_semantic.py --data ${{inputs.training_data}} --config ${{inputs.config}} --output ${{outputs.model}}",
#     environment="AzureML-pytorch-2.0@latest",
#     compute="gpu-cluster",  # runs on GPU!
# )

# @dsl.pipeline(
#     default_compute="cpu-cluster",
#     description="Spine segmentation training pipeline",
# )
# def spine_pipeline(input_data, config_file):
#     prep = prepare_step(raw_data=input_data)
#     train = train_step(
#         training_data=prep.outputs.processed_data,
#         config=config_file,
#     )
#     return {"trained_model": train.outputs.model}

# # Submit the pipeline
# pipeline_job = spine_pipeline(
#     input_data=Input(type="uri_folder", path="azureml:spine-ct-scans:2"),
#     config_file=Input(type="uri_file", path="./config/spine_semantic_new_v8.yaml"),
# )
# submitted = ml_client.jobs.create_or_update(
#     pipeline_job, experiment_name="spine-training"
# )
# print(f"Pipeline URL: {submitted.studio_url}")

print("Azure ML Pipelines:")
print("  - Define steps as components (reusable)")
print("  - Each step can run on different compute (CPU vs GPU)")
print("  - Data flows between steps automatically")
print("  - Visible in Azure Portal with full lineage")

### DVC Pipelines vs Azure ML Pipelines

| Feature | DVC Pipelines | Azure ML Pipelines |
|---------|--------------|--------------------|
| Definition | `dvc.yaml` (YAML) | `@dsl.pipeline` (Python) |
| Runs on | Your machine / any server | Azure compute (cloud) |
| GPU access | Your own GPU | Azure GPU VMs (pay per use) |
| Smart rerun | Yes (`dvc repro` skips unchanged) | Manual (resubmit jobs) |
| Visualization | `dvc dag` (terminal) | Azure Portal (rich UI) |
| Cost | Free | Pay for compute time |
| Scheduling | Cron (DIY) | Built-in scheduling |
| Lineage | `dvc.lock` | Azure ML lineage graph |

## 6. Compute Options

You have Azure credits. Here's what you can use:

| Compute type | What it is | When to use | Cost |
|-------------|-----------|-------------|------|
| **Compute Instance** | Single VM | Notebooks, experiments | Per hour (even idle) |
| **Compute Cluster** | Auto-scaling VMs | Training jobs | Per hour (scales to 0) |
| **Serverless** | Fully managed | Any job | Per job |
| **Attached** | Your own machine | Use existing hardware | Free (your hardware) |

In [ ]:
# === AZURE ML: Create a compute cluster ===

# from azure.ai.ml.entities import AmlCompute

# # GPU cluster for training (scales to 0 when idle = no cost)
# gpu_cluster = AmlCompute(
#     name="gpu-cluster",
#     type="amlcompute",
#     size="Standard_NC6s_v3",  # 1x V100 GPU
#     min_instances=0,           # scales to 0 when idle
#     max_instances=2,           # max 2 VMs
#     idle_time_before_scale_down=300,  # 5 min idle → scale down
# )
# ml_client.compute.begin_create_or_update(gpu_cluster).result()

# # Or use serverless (no cluster setup at all)
# # Just set compute="serverless" in your pipeline steps

print("Tip: Set min_instances=0 so the cluster scales down when idle.")
print("You only pay when training is running.")

## 7. Deployment (Serving Models)

Azure ML can deploy your model as an HTTP endpoint — useful for the inference pipeline.

Currently your inference runs in Docker locally. Azure ML can host it in the cloud.

In [ ]:
# === AZURE ML: Deploy a model ===

# from azure.ai.ml.entities import (
#     ManagedOnlineEndpoint,
#     ManagedOnlineDeployment,
# )

# # 1. Create endpoint (the URL)
# endpoint = ManagedOnlineEndpoint(
#     name="spine-segmentation",
#     description="Spine segmentation inference",
#     auth_mode="key",
# )
# ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# # 2. Deploy model to endpoint
# deployment = ManagedOnlineDeployment(
#     name="v9",
#     endpoint_name="spine-segmentation",
#     model=ml_client.models.get("spine-semantic-swinunetr", version="2"),
#     instance_type="Standard_DS3_v2",
#     instance_count=1,
# )
# ml_client.online_deployments.begin_create_or_update(deployment).result()

# # 3. Send traffic to deployment
# endpoint.traffic = {"v9": 100}
# ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("Deployment options:")
print("  Current:  Docker container on local/NAS machine")
print("  Azure ML: Managed endpoint with auto-scaling")
print("  Benefit:  Blue/green deployments (v8 → v9 with traffic splitting)")

### Blue/Green Deployment

Deploy a new model version without downtime:

```
Step 1: Deploy v9 alongside v8
  endpoint traffic: v8=100%, v9=0%

Step 2: Test v9, shift some traffic
  endpoint traffic: v8=80%, v9=20%

Step 3: v9 validated, full switch
  endpoint traffic: v8=0%, v9=100%

Step 4: Remove v8
```

This is safer than rebuilding a Docker image every time.

## 8. Full Lineage in Azure ML

When everything is connected, Azure ML shows the complete chain:

```
Data Asset: spine-ct-scans v2
    │
    ▼
Pipeline Job: spine-training-job-2024
    ├── Step 1: prepare_data
    ├── Step 2: train_model (MLflow logs: lr=0.001, dice=0.891)
    └── Step 3: evaluate
    │
    ▼
Model: spine-semantic-swinunetr v2
    │
    ▼
Endpoint: spine-segmentation (deployment: v9)
```

**For FDA 510(k):** This lineage graph is visible in Azure Portal. You can show an auditor exactly which data trained which model and where it's deployed.

## Recommendation: What to Use

**Start with (now):**

| Tool | For | Why |
|------|-----|-----|
| DVC | Data versioning | Free, portable, works with your Git workflow |
| MLflow | Experiment tracking | Free, and it's the same API Azure ML uses |

**Add later (when ready for cloud training):**

| Tool | For | Why |
|------|-----|-----|
| Azure Blob Storage | DVC remote | Replace NAS with cloud (you have credits) |
| Azure ML Compute | GPU training | Don't need to buy GPUs |
| Azure ML Pipelines | Automated training | Schedule retraining on new data |
| Azure ML Endpoints | Inference API | Replace Docker-on-NAS with managed endpoint |

**The migration path is smooth:**
1. Your DVC workflow stays the same — just change the remote to Azure Blob
2. Your MLflow code stays the same — just change the tracking URI to Azure ML
3. Your training code stays the same — just submit it as an Azure ML job

## Quick Reference: Getting Started Checklist

When you're ready to set up Azure ML:

```bash
# 1. Install Azure CLI
brew install azure-cli          # macOS

# 2. Login
az login

# 3. Create resources (one-time)
az group create --name maia-ml --location westeurope
az ml workspace create --name maia-workspace --resource-group maia-ml

# 4. Install Python SDK
pip install azure-ai-ml azure-identity azureml-mlflow

# 5. Download config
# Azure Portal → ML Workspace → Overview → Download config.json

# 6. Connect DVC to Azure Blob
pip install dvc-azure
dvc remote modify myremote url azure://maia-data-container/dvc-storage

# 7. Connect MLflow to Azure ML
# (see code example in section 3 above)
```

## Summary

| Concept | DVC + MLflow (open source) | Azure ML Studio (managed) |
|---------|--------------------------|---------------------------|
| Data versioning | `dvc add` → `.dvc` file | `Data()` → Data Asset |
| Experiment tracking | `mlflow.log_*()` | Same `mlflow.log_*()` |
| Model registry | `mlflow.register_model()` | `ml_client.models.create_or_update()` |
| Pipelines | `dvc.yaml` + `dvc repro` | `@dsl.pipeline` + submit job |
| Compute | Your machine | Azure GPU VMs (pay per use) |
| Deployment | Docker container (DIY) | Managed endpoint (auto-scaling) |
| Lineage | Git log + DVC + MLflow | Azure Portal lineage graph |

**Key takeaway:** Azure ML is not a replacement for DVC + MLflow — it's an upgrade path. Start with open source tools locally, then progressively move to Azure ML for compute, deployment, and a nicer UI. Your MLflow code works in both.